In [1]:
import pandas as pd
import numpy as np

In [2]:
order_reviews = pd.read_csv("../Data/order_reviews_dataset.csv")

In [3]:
order_reviews.head()

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,NaN,NaN,2018-01-18 00:00:00,2018-01-18 21:46:59
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,NaN,NaN,2018-03-10 00:00:00,2018-03-11 03:05:13
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,NaN,NaN,2018-02-17 00:00:00,2018-02-18 14:36:24
3,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,NaN,Recebi bem antes do prazo estipulado.,2017-04-21 00:00:00,2017-04-21 22:02:06
4,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,NaN,Parabéns lojas lannister adorei comprar pela I...,2018-03-01 00:00:00,2018-03-02 10:26:53


In [4]:
order_reviews.shape

(99224, 7)

In [5]:
order_reviews.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99224 entries, 0 to 99223
Data columns (total 7 columns):
 #   Column                   Non-Null Count  Dtype 
---  ------                   --------------  ----- 
 0   review_id                99224 non-null  object
 1   order_id                 99224 non-null  object
 2   review_score             99224 non-null  int64 
 3   review_comment_title     11568 non-null  object
 4   review_comment_message   40977 non-null  object
 5   review_creation_date     99224 non-null  object
 6   review_answer_timestamp  99224 non-null  object
dtypes: int64(1), object(6)
memory usage: 5.3+ MB


In [6]:
order_reviews.isna().sum()

review_id                      0
order_id                       0
review_score                   0
review_comment_title       87656
review_comment_message     58247
review_creation_date           0
review_answer_timestamp        0
dtype: int64

In [7]:
order_reviews.duplicated().sum()

np.int64(0)

In [8]:
# data type conversions
order_reviews["review_creation_date"] = pd.to_datetime(
    order_reviews["review_creation_date"]
)

order_reviews["review_answer_timestamp"] = pd.to_datetime(
    order_reviews["review_answer_timestamp"]
)

In [9]:
order_reviews.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99224 entries, 0 to 99223
Data columns (total 7 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   review_id                99224 non-null  object        
 1   order_id                 99224 non-null  object        
 2   review_score             99224 non-null  int64         
 3   review_comment_title     11568 non-null  object        
 4   review_comment_message   40977 non-null  object        
 5   review_creation_date     99224 non-null  datetime64[ns]
 6   review_answer_timestamp  99224 non-null  datetime64[ns]
dtypes: datetime64[ns](2), int64(1), object(4)
memory usage: 5.3+ MB


In [10]:
# primary key
order_reviews["review_id"].duplicated().sum()

np.int64(814)

In [11]:
# we have to use combination of keys for primary key
order_reviews.duplicated(
    subset=["review_id", "order_id"]
).sum()

np.int64(0)

In [12]:
# validating review score --> Allowed scores are 1–5.
order_reviews["review_score"].value_counts().sort_index()

review_score
1    11424
2     3151
3     8179
4    19142
5    57328
Name: count, dtype: int64

**FEATURE ENGINEERING**

In [14]:
order_reviews["has_review_title"] = (
    order_reviews["review_comment_title"]
    .notna()
)

order_reviews["has_review_message"] = (
    order_reviews["review_comment_message"]
    .notna()
)

In [15]:
# how quickly the marketplace responded
order_reviews["review_response_days"] = (
    order_reviews["review_answer_timestamp"]
    - order_reviews["review_creation_date"]
).dt.days

In [17]:
order_reviews[
    order_reviews["review_response_days"] < 0
]

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp,has_review_title,has_review_message,review_response_days


In [18]:
# Useful for sentiment-style analysis.

order_reviews["review_title_length"] = (
    order_reviews["review_comment_title"]
    .fillna("")
    .str.len()
)

order_reviews["review_message_length"] = (
    order_reviews["review_comment_message"]
    .fillna("")
    .str.len()
)

In [19]:
def rating_category(score):
    if score >= 4:
        return "Positive"
    elif score == 3:
        return "Neutral"
    else:
        return "Negative"

order_reviews["rating_category"] = (
    order_reviews["review_score"]
    .apply(rating_category)
)

In [20]:
# Extract Date Features
order_reviews["review_year"] = (
    order_reviews["review_creation_date"].dt.year
)

order_reviews["review_month"] = (
    order_reviews["review_creation_date"].dt.month
)

order_reviews["review_day_name"] = (
    order_reviews["review_creation_date"].dt.day_name()
)

In [21]:
order_reviews.head()

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp,has_review_title,has_review_message,review_response_days,review_title_length,review_message_length,rating_category,review_year,review_month,review_day_name
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,NaN,NaN,2018-01-18,2018-01-18 21:46:59,False,False,0,0,0,Positive,2018,1,Thursday
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,NaN,NaN,2018-03-10,2018-03-11 03:05:13,False,False,1,0,0,Positive,2018,3,Saturday
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,NaN,NaN,2018-02-17,2018-02-18 14:36:24,False,False,1,0,0,Positive,2018,2,Saturday
3,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,NaN,Recebi bem antes do prazo estipulado.,2017-04-21,2017-04-21 22:02:06,False,True,0,0,37,Positive,2017,4,Friday
4,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,NaN,Parabéns lojas lannister adorei comprar pela I...,2018-03-01,2018-03-02 10:26:53,False,True,1,0,100,Positive,2018,3,Thursday


In [22]:
import mysql.connector
from urllib.parse import quote_plus
from sqlalchemy import create_engine
print("Installed Successfully!")

Installed Successfully!


In [23]:
# MySQL credentials
username = "root"
password = quote_plus("Sanket@1024")
host = "localhost"
port = "3306"
database = "E_commerce"
# Create connection
engine = create_engine(
    f"mysql+pymysql://{username}:{password}@{host}:{port}/{database}"
)

# Upload DataFrame to MySQL
order_reviews.to_sql(
    "order_reviews",
    engine,
    if_exists="replace",
    index=False
)

print("Data uploaded successfully!")

Data uploaded successfully!
